In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
url = "https://madisonperfumery.com/product-category/perfume/page/2/?per_page=80&la_doing_ajax=true"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'html.parser')
    print("Successfully fetched and parsed the URL. Here's a snippet of the parsed HTML:")
    # Display a snippet of the HTML, for example, the title or first few elements
    if soup.title:
        print(f"Page Title: {soup.title.string}")
    else:
        print("No page title found.")
    # You can add more code here to extract specific data using soup.find(), soup.find_all(), etc.

except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully fetched and parsed the URL. Here's a snippet of the parsed HTML:
Page Title: PERFUME Archives | Page 2 of 9 | Madison Perfumery


In [ ]:
import requests
from bs4 import BeautifulSoup
import json

# # ---------------- INPUT ----------------
# url = "https://madisonperfumery.com/product-category/perfume/eau-de-parfum/"  # change if needed

# headers = {
#     "User-Agent": "Mozilla/5.0"
# }

# # ---------------- REQUEST ----------------
# response = requests.get(url, headers=headers)
# soup = BeautifulSoup(response.text, "html.parser")

products = []

# ---------------- PARSE ----------------
items = soup.select("li.product_item")

for item in items:
    try:
        # Product name + URL
        title_tag = item.select_one("h3.product_item--title a")
        name = title_tag.get_text(strip=True) if title_tag else None
        product_url = title_tag["href"] if title_tag else None

        # Image
        img_tag = item.select_one(".product_item--thumbnail img")
        image_url = img_tag["src"] if img_tag else None

        # Price (handles single or range)
        price_tag = item.select(".price .woocommerce-Price-amount bdi")
        prices = [p.get_text(strip=True) for p in price_tag]
        price = " - ".join(prices) if prices else None

        # Category
        category_tag = item.select_one(".product_item--category-link a")
        category = category_tag.get_text(strip=True) if category_tag else None

        # Description
        desc_tag = item.select_one(".item--excerpt p")
        description = desc_tag.get_text(strip=True) if desc_tag else None

        # Product ID + SKU (from button)
        button = item.select_one(".add_to_cart_button")
        product_id = button.get("data-product_id") if button else None
        sku = button.get("data-product_sku") if button else None

        products.append({
            "name": name,
            "product_url": product_url,
            "image_url": image_url,
            "price": price,
            "category": category,
            "description": description,
            "product_id": product_id,
            "sku": sku
        })

    except Exception as e:
        print("Error parsing item:", e)

# ---------------- OUTPUT ----------------
print(json.dumps(products, indent=2, ensure_ascii=False))

[
  {
    "name": "Yasat EDP WIDIAN",
    "product_url": "https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/widian-yasat-edp/",
    "image_url": "https://madisonperfumery.com/wp-content/uploads/Yasat-EDP-WIDIAN-300x300.jpg",
    "price": "1.750lei",
    "category": "EAU DE PARFUM",
    "description": "Inspired by the tranquil embrace of an island oasis, Yasat perfume blends the freshness of…",
    "product_id": "86977",
    "sku": "MX.003083"
  },
  {
    "name": "Alto Astral EDP BYREDO",
    "product_url": "https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/alto-astral-edp/",
    "image_url": "https://madisonperfumery.com/wp-content/uploads/ALTO-ASTRAL-41-300x300.jpg",
    "price": "860lei - 1.250lei",
    "category": "EAU DE PARFUM",
    "description": "An embodiment of positive energy, Alto Astral Eau de Parfum takes its name from a…",
    "product_id": "86181",
    "sku": "MX.003041"
  },
  {
    "name": "Max Richter 01 EDT COMME DES GARCONS",
    "

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# ---------------- CONFIG ----------------
BASE_URL = "https://madisonperfumery.com/product-category/home"
FIRST_PAGE_URL = f"{BASE_URL}/?per_page=80&la_doing_ajax=true"
PAGE_URL_TEMPLATE = BASE_URL + "/page/{page}/?per_page=80&la_doing_ajax=true"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

OUTPUT_FILE = "home_products.json"

# ---------------- FUNCTION: PARSE PRODUCTS ----------------
def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    items = soup.select("li.product_item")

    products = []

    for item in items:
        try:
            title_tag = item.select_one("h3.product_item--title a")
            name = title_tag.get_text(strip=True) if title_tag else None
            product_url = title_tag["href"] if title_tag else None

            img_tag = item.select_one(".product_item--thumbnail img")
            image_url = img_tag["src"] if img_tag else None

            price_tag = item.select(".price .woocommerce-Price-amount bdi")
            prices = [p.get_text(strip=True) for p in price_tag]
            price = " - ".join(prices) if prices else None

            category_tag = item.select_one(".product_item--category-link a")
            category = category_tag.get_text(strip=True) if category_tag else None

            desc_tag = item.select_one(".item--excerpt p")
            description = desc_tag.get_text(strip=True) if desc_tag else None

            button = item.select_one(".add_to_cart_button")
            product_id = button.get("data-product_id") if button else None
            sku = button.get("data-product_sku") if button else None

            products.append({
                "name": name,
                "product_url": product_url,
                "image_url": image_url,
                "price": price,
                "category": category,
                "description": description,
                "product_id": product_id,
                "sku": sku
            })

        except Exception as e:
            print("Error parsing item:", e)

    return products


# ---------------- FUNCTION: FETCH PAGE ----------------
def fetch_page(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {url} -> {e}")
        return None


# ---------------- MAIN SCRAPER ----------------
def scrape_all():
    all_products = []
    page = 1

    while True:
        if page == 1:
            url = FIRST_PAGE_URL
        else:
            url = PAGE_URL_TEMPLATE.format(page=page)

        print(f"Fetching page {page}: {url}")

        html = fetch_page(url)
        if not html:
            print("Stopping due to fetch failure.")
            break

        products = parse_products(html)
        count = len(products)

        print(f"Found {count} products on page {page}")

        if count == 0:
            print("No products found. Stopping.")
            break

        all_products.extend(products)

        # ✅ Stop condition
        if count < 80:
            print("Last page reached (less than 80 products).")
            break

        page += 1
        time.sleep(1)  # polite delay

    return all_products


# ---------------- RUN ----------------
if __name__ == "__main__":
    data = scrape_all()

    print(f"\nTotal products scraped: {len(data)}")

    # Save to file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")

Fetching page 1: https://madisonperfumery.com/product-category/home/?per_page=80&la_doing_ajax=true
Found 80 products on page 1
Fetching page 2: https://madisonperfumery.com/product-category/home/page/2/?per_page=80&la_doing_ajax=true
Found 80 products on page 2
Fetching page 3: https://madisonperfumery.com/product-category/home/page/3/?per_page=80&la_doing_ajax=true
Found 80 products on page 3
Fetching page 4: https://madisonperfumery.com/product-category/home/page/4/?per_page=80&la_doing_ajax=true
